# ESIS Mobile V2 — Kaggle APK Build

**Edge Survival Intelligence System** — Gemma 4 powered crisis intervention for people experiencing homelessness.

## What this builds
- Gemma 4 (HuggingFace) powered intervention plans
- Housing track assignment across 9 program tracks
- Advocacy packet generation (one-page summary, advocate script, referral handoff)
- Police interaction log
- Community ping (real-time crisis location sharing)

## Instructions
1. Upload the `mobile/` directory as a Kaggle Dataset named **`esis-mobile-v2`**
2. Add it as an input to this notebook under `/kaggle/input/esis-mobile-v2/`
3. **Run All** — build takes ~10–15 minutes
4. Download **`esis-v2-debug.apk`** from the Output panel (right side)

Produces a debug APK installable on any Android phone (enable *Install unknown apps* in settings).

In [ ]:
%%bash
# Cell 1 — Install Node 20 and Java 17
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - 2>/dev/null
sudo apt-get install -y nodejs 2>/dev/null | tail -3

node --version
npm --version

sudo apt-get install -y openjdk-17-jdk 2>/dev/null | tail -3
java -version

In [ ]:
%%bash
# Cell 2 — Install Android SDK command-line tools
cd /tmp

wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip \
  -O cmdtools.zip
unzip -q cmdtools.zip -d android-sdk
mkdir -p android-sdk/cmdline-tools/latest
mv android-sdk/cmdline-tools/cmdline-tools/* android-sdk/cmdline-tools/latest/

export ANDROID_HOME=/tmp/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools

yes | sdkmanager --licenses 2>/dev/null | tail -1
sdkmanager "platform-tools" "build-tools;34.0.0" "platforms;android-34" 2>&1 | tail -5
echo "Android SDK ready."

In [ ]:
%%bash
# Cell 3 — Copy mobile/ source and install npm deps
cp -r /kaggle/input/esis-mobile-v2/mobile /tmp/esis-mobile
cd /tmp/esis-mobile

npm install --legacy-peer-deps 2>&1 | tail -8
echo "npm install done"

In [ ]:
%%bash
# Cell 4 — TypeScript check (catches type errors before the slow Gradle build)
cd /tmp/esis-mobile

npx tsc --noEmit 2>&1
echo "TypeScript check complete (exit $?)"

In [ ]:
%%bash
# Cell 5 — Prebuild Android project with Expo
export ANDROID_HOME=/tmp/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools

cd /tmp/esis-mobile

# Expo CLI is bundled with the expo package — no global install needed
npx expo prebuild --platform android --clean 2>&1 | tail -15
echo "Prebuild complete"

In [ ]:
%%bash
# Cell 6 — Build debug APK with Gradle
export ANDROID_HOME=/tmp/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64

cd /tmp/esis-mobile/android

chmod +x gradlew
./gradlew assembleDebug --no-daemon 2>&1 | tail -25
echo "Build complete"

In [ ]:
import shutil, os

src = '/tmp/esis-mobile/android/app/build/outputs/apk/debug/app-debug.apk'
dst = '/kaggle/working/esis-v2-debug.apk'

shutil.copy(src, dst)
size_mb = os.path.getsize(dst) / 1024 / 1024
print(f'APK ready: {dst} ({size_mb:.1f} MB)')
print('Download from the Output panel on the right →')
print()
print('Install on Android: Settings → Apps → Install unknown apps → enable for your browser/Files app')